# Step 3: encode_batch tests
`encode_batch` lives in `03_encode_batch_step.py`; this notebook only tests it.

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from importlib import import_module
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import torch
from torch import nn

from helpers.pairs import load_pairs

_vocab_step = import_module("siamese_model.01_vocab_step")       # digit-prefixed filenames -> can't use `from ... import`
build_vocab = _vocab_step.build_vocab
PAD_ID, UNK_ID = _vocab_step.PAD_ID, _vocab_step.UNK_ID
encode_batch = import_module("siamese_model.03_encode_batch_step").encode_batch

In [3]:
ot_train, ot_train_label = load_pairs("overture", "train")   # only the train split: the vocab is built from train

vocab = build_vocab(ot_train)
print(len(vocab), vocab)

40 {'<pad>': 0, '<unk>': 1, ' ': 2, 'e': 3, 't': 4, 'a': 5, 's': 6, 'r': 7, 'n': 8, 'o': 9, 'i': 10, 'd': 11, '1': 12, 'l': 13, '2': 14, '0': 15, 'm': 16, '5': 17, 'v': 18, '3': 19, 'w': 20, '4': 21, 'h': 22, 'p': 23, '6': 24, 'c': 25, '7': 26, '9': 27, '8': 28, 'b': 29, 'u': 30, 'g': 31, 'k': 32, 'y': 33, 'f': 34, 'x': 35, 'q': 36, '-': 37, 'j': 38, 'z': 39}


In [4]:
# uneven lengths: the short string is right-padded to the longest in the batch
ids, lengths = encode_batch(["g", "g" * 20], vocab)
assert ids.shape == (2, 20) and ids.dtype == torch.long
assert lengths.tolist() == [1, 20]
assert (ids[0, 1:] == PAD_ID).all()

# lengths agree with the number of non-pad ids (the mask the encoder will use)
assert (lengths == (ids != PAD_ID).sum(1)).all()

# unseen character -> unk, never pad (pad would be masked out as if it weren't there)
ids, lengths = encode_batch(["a@b"], vocab)
assert ids[0, 1] == UNK_ID and lengths.item() == 3

# normalised before lookup: uppercase and extra whitespace encode the same as the clean form
assert torch.equal(encode_batch(["  12  MAIN ST "], vocab)[0], encode_batch(["12 main st"], vocab)[0])

# strings over 85 characters are truncated to 85
ids, lengths = encode_batch(["a" * 110, "b"], vocab)
assert ids.shape == (2, 85) and lengths.tolist() == [85, 1]
print("encode_batch: all passed")

encode_batch: all passed
